# Multiple linear regression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
In real life a house price depends on more than its size: the number of rooms, the age of the building and the distance to the metro matter too. With several features, the linear model is:

$$\hat y=b+w_1x_1+w_2x_2+\dots+w_dx_d$$

With two features the model is a **plane** instead of a line:

> [With two features, the linear model is a plane in 3-D space.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/lin_multi.png)

*With two features, the linear model is a plane in 3-D space.*

## The matrix form
Add a column of ones to the data; then $X$ is an $n\times(d+1)$ matrix and $\mathbf w$ is the coefficient vector:

$$\hat{\mathbf y}=X\mathbf w,\qquad \mathbf w^*=(X^\top X)^{-1}X^\top\mathbf y$$

The second formula is the **normal equation**. With very many features, we use gradient descent (next lesson) instead of inverting the matrix.

| Number of features | Suitable method |
|---|---|
| Few up to a few thousand | Normal equation or `LinearRegression` |
| Very many, or huge data | Gradient descent, `SGDRegressor` |

## Interpreting the coefficients
$w_j$ means: "if $x_j$ grows by one unit **and the other features stay the same**, the prediction changes by $w_j$."

> Compare coefficients only after **scaling** the features. The coefficient of "size in meters" and "size in centimeters" differ by a factor of 100, yet they are equally important.

## Example: the diabetes data
The Diabetes data has ten features (age, sex, BMI, blood pressure and six blood tests). The target is disease progression one year later.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

model = LinearRegression().fit(X_tr, y_tr)
coef = pd.Series(model.coef_, index=X.columns).sort_values()
print(coef.round(1))
print("Test MAE:", round(mean_absolute_error(y_te, model.predict(X_te)), 1))
print("Test R2 :", round(r2_score(y_te, model.predict(X_te)), 3))

# The same answer from the normal equation
A = np.c_[np.ones(len(X_tr)), X_tr]
w = np.linalg.solve(A.T @ A, A.T @ y_tr)
print("Same intercept?", np.isclose(w[0], model.intercept_))



## Multicollinearity
If two features depend strongly on each other (for example size and number of rooms), the model cannot decide which one deserves the effect. The coefficients become unstable and may even flip sign, although the predictions stay good.

| Symptom | How to detect | Remedy |
|---|---|---|
| Coefficients change with small data changes | Correlation matrix | Drop one of the features |
| A coefficient has a nonsensical sign | VIF above 5 to 10 | Ridge (next lessons) |
| | | PCA |



In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

X = load_diabetes(as_frame=True).data
print(X.corr().round(2).loc[["s1", "s2", "s3", "s4"], ["s1", "s2", "s3", "s4"]])

# Variance inflation factor: VIF_j = 1 / (1 - R2 of x_j predicted from the other features)
vif = {}
for col in X.columns:
    others = X.drop(columns=col)
    r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
    vif[col] = 1 / (1 - r2)
print(pd.Series(vif).round(1).sort_values(ascending=False))



## Categorical features
A column like "city" is not a number. **One-hot** encoding makes a 0/1 column for each city. We drop one column (`drop="first"`) to avoid perfect collinearity.



In [ ]:
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "size": [80, 120, 95, 150, 60, 110],
    "city": ["Tehran", "Shiraz", "Tehran", "Tabriz", "Shiraz", "Tabriz"],
    "price": [9.5, 7.8, 10.6, 9.0, 4.1, 6.9],
})
prep = make_column_transformer((OneHotEncoder(drop="first"), ["city"]), remainder="passthrough")
model = make_pipeline(prep, LinearRegression()).fit(df[["size", "city"]], df["price"])
print(prep.get_feature_names_out())
print(model[-1].coef_.round(3))
print(model.predict(pd.DataFrame({"size": [100], "city": ["Tehran"]})).round(2))

### Exercise

On the diabetes data, use only bmi, bp and s5. How much does the test R² drop?

In [ ]:
# Your answer here